In [154]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset, random_split
import torchvision.datasets as datasets
import os
%matplotlib inline

In [155]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [156]:
def load_data(Jd, l, num_conf, T, num_temps, batch_size, shuffle_opt, opt='train'):
    datasets = []
    for j in range(num_temps):
        path = f'data_spins/{Jd}_{opt}/spins_{l}_{T[j]}.npy'
        #if os.path.isfile(path):
        with open(path, 'rb') as f:
            x = np.load(f)   
        tensor_x = torch.Tensor(x).unsqueeze(1)
        path = f'data_spins/{Jd}_{opt}/answ_{l}_{T[j]}.npy'
        #if os.path.isfile(path):
        with open(path, 'rb') as f:
            y = np.load(f)
        tensor_y = torch.from_numpy(y).type(torch.float32)

        datasets.append(TensorDataset(tensor_x, tensor_y))


    dataset = torch.utils.data.ConcatDataset(datasets)

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle_opt)

In [157]:
class Net(nn.Module):
    def __init__(self, l):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, 2)
        self.pool = nn.MaxPool2d(2, 2)
        self.act_hid = nn.ReLU()
        self.fc1 = nn.Linear(64*int(l/2-1)*int(l/2-1), 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool(x)
        x = self.act_hid(x)
        x = x.view(-1, 64*int(l/2-1)*int(l/2-1))
        x = self.fc1(x)
        x = self.act_hid(x)
        x = self.fc2(x)
        return x

In [158]:
def train(l, train_dataloader, num_epoch, criterion, batch_size):
    model = Net(l)
    optimizer = torch.optim.Adam(model.parameters(), lr=1.0e-4)
    act = nn.Sigmoid()

    for epoch in range(num_epoch):  
        running_loss = 0.0
        accuracy = 0.0
        pbar = tqdm(enumerate(train_dataloader), total=len(train_dataloader))
        for i, data in pbar:
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)  
            
            model.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            #outputs = act(outputs)

            outputs = outputs.squeeze(1) # к одной размерности с labels

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            accuracy += (batch_size - sum(abs(labels - act(outputs)))).float().mean()

            pbar.set_description(
                    f"Loss: {running_loss/((i+1)*batch_size)} "
                    f"Accuracy: {accuracy * 100  / ((i+1)*batch_size)}"
            )

    print('Training completed')
    return model

In [159]:
def testing(model, test_dataloader, criterion, batch_size):
    outp = []
    errors = []
    accuracy = 0.0
    act = nn.Sigmoid()
    with torch.no_grad():
        for i, data in enumerate(test_dataloader):
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)  
            model.to(device)
            outputs = model(inputs)
            #outputs = act(outputs)
            outputs = outputs.squeeze(1)
            outp.append(act(outputs).item())
            loss = criterion(outputs, labels)
            errors.append(loss.item())

            accuracy += (1 - sum(abs(labels - act(outputs)))).float().mean()

    print("Accuracy = {}".format(accuracy / len(test_dataloader)))
    return outp, errors

In [160]:
roots = [2.2691853142129728, 2.104982167992544, 1.932307699120554, 1.749339162933206, 1.5536238493280832, 1.34187327905057, 1.109960313758399, 0.8541630993606272, 0.5762735442012712, 0.2885386111960936, 0.03198372863548067]
jds = [0.0, -0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1.0]
get_crit_T = dict(zip(jds, roots))

In [161]:
### training ###

def train_and_save(Jd, l, num_temps):
    num_conf_tr = 2048
    num_conf_ts = 512
    num_epoch = 1

    T_c = get_crit_T[Jd]
    T = np.round(np.linspace(T_c - 0.3, T_c + 0.3, num_temps), 4)

    criterion = nn.BCEWithLogitsLoss()     

    train_dataloader = load_data(Jd, l, num_conf_tr, T, num_temps, batch_size=4, shuffle_opt=True, opt='train')
    print(f'Start training for L = {l}')
    model = train(l, train_dataloader, num_epoch, criterion, batch_size=4)

    PATH = f'models/{l}_{Jd}_{T[0]}_{T[-1]}_{num_temps}.pt'
    torch.save(model.state_dict(), PATH)

In [170]:
L = [80]
Jd = 0.0
num_temps = 20
for l in L:
    train_and_save(Jd, l, num_temps)

Start training for L = 80


Loss: 0.02416447896184893 Accuracy: 95.53541564941406: 100%|██████████| 10240/10240 [07:43<00:00, 22.10it/s] 


Training completed


In [163]:
l = 80
Jd = -1.0
num_temps = 100
T_c = get_crit_T[Jd]
T = np.linspace(T_c - 0.3, T_c + 0.3, num_temps)
T = np.round(T, 4)

opt = 'train'
T_miss = []
for j in range(num_temps):
        path = f'data_spins/{Jd}_{opt}/spins_{l}_{T[j]}.npy'
        if not os.path.isfile(path):
            T_miss.append(T[j])
print(T_miss, len(T_miss))

opt = 'test'
T_miss = []
for j in range(num_temps):
        path = f'data_spins/{Jd}_{opt}/spins_{l}_{T[j]}.npy'
        if not os.path.isfile(path):
            T_miss.append(T[j])
print(T_miss, len(T_miss))

[-0.268, -0.262, -0.2559, -0.2498, -0.2438, -0.2377, -0.2317, -0.2256, -0.2195, -0.2135, -0.2074, -0.2013, -0.1953, -0.1892, -0.1832, -0.1771, -0.171, -0.165, -0.1589, -0.1529, -0.1468, -0.1407, -0.1347, -0.1286, -0.1226, -0.1165, -0.1104, -0.1044, -0.0983, -0.0923, -0.0862, -0.0801, -0.0741, -0.068, -0.062, -0.0559, -0.0498, -0.0438, -0.0377, -0.0317, -0.0256, -0.0195, -0.0135, -0.0074, -0.0013, 0.0047, 0.0108, 0.0168, 0.0229, 0.029, 0.035, 0.0411, 0.0471, 0.0532, 0.0593, 0.0653, 0.0714, 0.0774, 0.0835, 0.0896, 0.0956, 0.1017, 0.1077, 0.1138, 0.1199, 0.1259, 0.132, 0.138, 0.1441, 0.1502, 0.1562, 0.1623, 0.1683, 0.1744, 0.1805, 0.1865, 0.1926, 0.1987, 0.2047, 0.2108, 0.2168, 0.2229, 0.229, 0.235, 0.2411, 0.2471, 0.2532, 0.2593, 0.2653, 0.2714, 0.2774, 0.2835, 0.2896, 0.2956, 0.3017, 0.3077, 0.3138, 0.3199, 0.3259, 0.332] 100
[] 0


In [168]:
R = np.linspace(0.03, 3.5, 50)
R[1] - R[0]

0.07081632653061225

In [174]:
l = 80
Jd = 0.0
num_temps = 20
T_c = get_crit_T[Jd]
T = np.linspace(T_c - 0.3, T_c + 0.3, num_temps)
T = np.round(T, 4)

opt = 'train'
T_miss = []
for j in range(num_temps):
        path = f'data_spins/{Jd}_{opt}/spins_{l}_{T[j]}.npy'
        if not os.path.isfile(path):
            T_miss.append(T[j])
print(T_miss, len(T_miss))

opt = 'test'
T_miss = []
for j in range(num_temps):
        path = f'data_spins/{Jd}_{opt}/spins_{l}_{T[j]}.npy'
        if not os.path.isfile(path):
            T_miss.append(T[j])
print(T_miss, len(T_miss))

[] 0
[] 0


In [175]:
### testing ###

def get_errs_outs(Jd, l, num_temps):
    T_c = get_crit_T[Jd]
    T = np.linspace(T_c - 0.3, T_c + 0.3, num_temps)
    T = np.round(T, 4)

    num_conf_tr = 2048
    num_conf_ts = 512

    criterion = nn.BCEWithLogitsLoss()     
    
    print(f'Start testing for L = {l}')
    model = Net(l)
    T_c_ = get_crit_T[0.0]
    T_ = np.round(np.linspace(T_c_ - 0.3, T_c_ + 0.3, num_temps), 4)
    PATH = f'models/{l}_0.0_{T_[0]}_{T_[-1]}_{num_temps}.pt'

    model.load_state_dict(torch.load(PATH))
    model.eval()
    test_dataloader = load_data(Jd, l, num_conf_ts, T, num_temps, batch_size=1, shuffle_opt=False, opt='test')
    outp, errors = testing(model, test_dataloader, criterion, batch_size=1)
    return errors, outp

In [177]:
L = [80]
Jds = [-0.3, -0.5, -0.7, -0.9, -1.0]
num_temps = 20
for Jd in Jds: 
    for l in L:
        errs_outs = get_errs_outs(Jd, l, num_temps)
        np.save(f'data_errors/{Jd}_{l}_{num_temps}.npy', errs_outs[0])
        np.save(f'data_outputs/{Jd}_{l}_{num_temps}.npy', errs_outs[1])

Start testing for L = 80
Accuracy = 0.9673558473587036
Start testing for L = 80
Accuracy = 0.9659093618392944
Start testing for L = 80
Accuracy = 0.9593324661254883
Start testing for L = 80
Accuracy = 0.8914942741394043
Start testing for L = 80
Accuracy = 0.49999508261680603


In [12]:
from torchsummary import summary

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # PyTorch v0.4.0
l = 20
model = Net(l).to(device)

summary(model, (1, l, l))

Layer (type:depth-idx)                   Output Shape              Param #
├─Conv2d: 1-1                            [-1, 64, 19, 19]          320
├─MaxPool2d: 1-2                         [-1, 64, 9, 9]            --
├─ReLU: 1-3                              [-1, 64, 9, 9]            --
├─Linear: 1-4                            [-1, 64]                  331,840
├─ReLU: 1-5                              [-1, 64]                  --
├─Linear: 1-6                            [-1, 1]                   65
Total params: 332,225
Trainable params: 332,225
Non-trainable params: 0
Total mult-adds (M): 0.42
Input size (MB): 0.00
Forward/backward pass size (MB): 0.18
Params size (MB): 1.27
Estimated Total Size (MB): 1.45


Layer (type:depth-idx)                   Output Shape              Param #
├─Conv2d: 1-1                            [-1, 64, 19, 19]          320
├─MaxPool2d: 1-2                         [-1, 64, 9, 9]            --
├─ReLU: 1-3                              [-1, 64, 9, 9]            --
├─Linear: 1-4                            [-1, 64]                  331,840
├─ReLU: 1-5                              [-1, 64]                  --
├─Linear: 1-6                            [-1, 1]                   65
Total params: 332,225
Trainable params: 332,225
Non-trainable params: 0
Total mult-adds (M): 0.42
Input size (MB): 0.00
Forward/backward pass size (MB): 0.18
Params size (MB): 1.27
Estimated Total Size (MB): 1.45